# Как модель видит текст и что выбирает дальше · День −1

Два маленьких эксперимента, чтобы **пощупать** две вещи, о которых пойдёт речь на первой лекции: что такое токен и что значит «модель предсказывает следующий токен».

Здесь только «смотрите, как». «Почему так и что с этим делать» — на лекции.

Нужен ключ в `OPENROUTER_API_KEY` (переменная окружения или `.env` рядом).

In [1]:
import os, requests
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
KEY = os.getenv('OPENROUTER_API_KEY')
assert KEY, 'Положите ключ в OPENROUTER_API_KEY (см. День −3)'
HEAD = {'Authorization': f'Bearer {KEY}'}
URL = 'https://openrouter.ai/api/v1/chat/completions'
print('ключ найден')

ключ найден


## 1. Токены: чем модель меряет текст
Модель не видит буквы или слова — она видит **токены**, куски текста. Отправим одно и то же предложение по-английски и по-русски и сравним, во сколько токенов каждое обошлось (число берём из `usage.prompt_tokens`).

In [2]:
def prompt_tokens(text):
    r = requests.post(URL, headers=HEAD, json={
        'model': 'openai/gpt-4o-mini',
        'messages': [{'role': 'user', 'content': text}],
        'max_tokens': 1}, timeout=60).json()
    return r['usage']['prompt_tokens']

pairs = [
    ('The cat sat on the mat.',        'Кошка сидела на коврике.'),
    ('Artificial intelligence is here.', 'Искусственный интеллект уже здесь.'),
]
for en, ru in pairs:
    te, tr = prompt_tokens(en), prompt_tokens(ru)
    print(f'EN {te:2d} | RU {tr:2d}  ({tr/te:.1f}x)   {en}')
print()
print('Русский стоит дороже: кириллица режется на более мелкие токены.')

EN 14 | RU 17  (1.2x)   The cat sat on the mat.


EN 12 | RU 15  (1.2x)   Artificial intelligence is here.

Русский стоит дороже: кириллица режется на более мелкие токены.


Отсюда практическое следствие (разберём на лекции): **за токены платят**, вход и выход по-разному, и русский текст обходится дороже английского того же смысла: на коротких фразах примерно в 1,2 раза, а на длинных текстах и более старых токенизаторах разрыв доходит до полутора-двух раз.

## 2. Модель предсказывает следующий токен
В своей основе модель делает одно: смотрит на текст и выдаёт **распределение вероятностей** по тому, каким будет следующий токен. Попросим показать топ-5 вариантов (`logprobs`).

Сначала вопрос, где ответ почти однозначен.

In [3]:
def next_tokens(prompt, k=5):
    r = requests.post(URL, headers=HEAD, json={
        'model': 'openai/gpt-4o-mini',
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': 1, 'temperature': 0,
        'logprobs': True, 'top_logprobs': k}, timeout=60).json()
    out = r['choices'][0]['logprobs']['content'][0]['top_logprobs']
    for e in out:
        p = 2.718281828 ** e['logprob']
        bar = '#' * int(p * 40)
        print(f"  {e['token']!r:12s} {p*100:5.1f}%  {bar}")

print("Answer with one word. The opposite of 'hot' is")
next_tokens("Answer with one word. The opposite of 'hot' is")

Answer with one word. The opposite of 'hot' is


  'Cold'       100.0%  #######################################
  'cold'         0.0%  
  'Cool'         0.0%  
  ' cold'        0.0%  
  ' Cold'        0.0%  


Модель почти уверена — вся вероятность на одном варианте. Теперь вопрос без единственного правильного ответа:

In [4]:
print('Complete with one word. My favorite season of the year is')
next_tokens('Complete with one word. My favorite season of the year is')

Complete with one word. My favorite season of the year is


  'aut'         77.3%  ##############################
  'fall'        13.4%  #####
  'Aut'          8.1%  ###
  'Fall'         0.5%  
  'spring'       0.5%  


Здесь вероятность **размазана** по нескольким вариантам. Это и есть распределение, из которого модель делает выбор. Заметьте: слово `autumn` могло само распасться на токены (`aut` + `umn`) — вот что значит «модель видит токены, а не слова».

Практическое следствие (на лекции): ручка **temperature** управляет тем, насколько случайно модель выбирает из этого распределения. Ноль — почти всегда самый вероятный вариант, выше — больше разнообразия.

## 3. (Необязательно) Увидеть разбиение на токены глазами
Библиотека `tiktoken` показывает, как строка режется на токены. Установите её и запустите.

In [5]:
try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    for text in ['tokenization', 'токенизация', 'The cat sat.', 'Кошка сидела.']:
        pieces = [enc.decode([t]) for t in enc.encode(text)]
        print(f'{text:16s} -> ' + ' | '.join(pieces))
except ImportError:
    print('Установите:  pip install tiktoken   — и перезапустите эту ячейку,')
    print('чтобы увидеть, как текст режется на токены.')

Установите:  pip install tiktoken   — и перезапустите эту ячейку,
чтобы увидеть, как текст режется на токены.


---
**Что уносим на лекцию:** модель меряет текст токенами (и русский дороже), а в основе просто предсказывает следующий токен из распределения вероятностей. Из этих двух фактов на первой лекции вырастут деньги, температура и почти всё остальное.